## **1. Persiapan**

In [ ]:
# Install library yang dibutuhkan
!pip install pandas openpyxl nltk Sastrawi gensim pyLDAvis

## **2. Ubah data**

In [ ]:
import pandas as pd

# 1. Loaad data
file_excel = '1DATA_FIX.xlsx'
df = pd.read_excel(file_excel)

# 2. Menyimpan data ke CSV
file_csv = '1DATA_FIX.csv'
df.to_csv(file_csv, index=False)

df = pd.read_csv(file_csv)

# Menampilkan info data untuk memastikan jumlah 235 dokumen
print(f"Jumlah dokumen: {len(df)}")

# 3. Menggabungkan kolom 'judul' dan 'abstrak'
df['Judul'] = df['Judul'].fillna('')
df['Abstrak'] = df['Abstrak'].fillna('')

df['teks_gabungan'] = df['Judul'] + " " + df['Abstrak']

# Menampilkan beberapa data awal
df[['Judul', 'Abstrak', 'teks_gabungan']].head()

Jumlah dokumen: 235


,Judul,Abstrak,teks_gabungan
0,Perbandingan Metode Ceramah dan Metode Tutor ...,Abstrak Penelitian ini bertujuan untuk menguji...,Perbandingan Metode Ceramah dan Metode Tutor ...
1,Analisis Kemampuan Berpikir Kritis Siswa Melal...,Penelitian ini bertujuan untuk mengetahui (1) ...,Analisis Kemampuan Berpikir Kritis Siswa Melal...
2,Efektivitas Model Pembelajaran Guided Inquiry ...,Penelitian ini bertujuan untuk mengetahui (1) ...,Efektivitas Model Pembelajaran Guided Inquiry ...
3,Implementasi Model Pembelajaran Team Assisted ...,Penelitian ini bertujuan untuk mengetahui perb...,Implementasi Model Pembelajaran Team Assisted ...
4,Pengaruh Penggunaan Media Pembelajaran Modul B...,Penelitian ini bertujuan untuk membuktikan ada...,Pengaruh Penggunaan Media Pembelajaran Modul B...


## **3. Preprocessing**

In [ ]:
import re
import nltk
from nltk.tokenize import word_tokenize
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

nltk.download('punkt')
nltk.download('punkt_tab')

# Inisialisasi Sastrawi
stopword_factory = StopWordRemoverFactory()

# Get default stopwords from Sastrawi
default_sastrawi_stopwords = stopword_factory.get_stop_words()

# Initialize stopwords_id as an empty list
stopwords_id = []

# Extend stopwords_id with default Sastrawi stopwords
stopwords_id.extend(default_sastrawi_stopwords)

# Extend stopwords_id with additional custom stopwords
stopwords_id.extend([
    "penelitian", "hasil", "metode", "data", "analisis",
    "menggunakan", "berdasarkan", "dilakukan", "dapat",
    "pengaruh", "signifikan", "positif", "negatif",
    "mahasiswa", "siswa", "peserta", "didik",
    "pembelajaran", "kelas", "universitas", "sebelas", "maret",
    "studi", "tujuan", "manfaat", "proses", "sistem",
    "aplikasi", "website", "media", "evaluasi", "efektivitas",
    "sangat", "terdapat", "memfasilitasi", "sebagai", "mempunyai",
    "berupa", "selain", "antar", "antara", "pihak", "para",
    "uns", "surakarta", "jawa", "tengah", "skripsi", "tugas", "akhir",
    "membangun", "merancang", "penerapan", "implementasi",
    "dikarenakan", "sehingga", "kemudian", "dengan", "untuk",
    "dari", "ini", "itu", "tersebut", "adalah", "yaitu", "eve", "ng", "er", "me", "strong"
    ])

stemmer_factory = StemmerFactory()
stemmer = stemmer_factory.create_stemmer()

def preprocess_text(text):
    # Lowercase
    text = str(text).lower()
    # Hapus angka dan tanda baca
    text = re.sub(r'[^a-z\s]', ' ', text)
    # Tokenisasi
    tokens = word_tokenize(text)
    # Hapus stopwords & kata yang terlalu pendek (< 3 huruf)
    tokens = [word for word in tokens if word not in stopwords_id and len(word) > 2]
    # Stemming
    tokens = [stemmer.stem(word) for word in tokens]
    # Filter token kosong setelah stemming
    tokens = [word for word in tokens if word]
    return tokens

# Menerapkan preprocessing
print("Memulai preprocessing teks...")
df['teks_bersih'] = df['teks_gabungan'].apply(preprocess_text)
print("Preprocessing selesai!")

df[['teks_gabungan', 'teks_bersih']].head()

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Memulai preprocessing teks...
Preprocessing selesai!


,teks_gabungan,teks_bersih
0,Perbandingan Metode Ceramah dan Metode Tutor ...,"[banding, ceramah, tutor, baya, minat, ajar, m..."
1,Analisis Kemampuan Berpikir Kritis Siswa Melal...,"[mampu, pikir, kritis, lalu, model, problem, b..."
2,Efektivitas Model Pembelajaran Guided Inquiry ...,"[model, guided, inquiry, tinjau, mampu, pikir,..."
3,Implementasi Model Pembelajaran Team Assisted ...,"[model, team, assisted, individualization, tai..."
4,Pengaruh Penggunaan Media Pembelajaran Modul B...,"[guna, modul, bas, teks, video, ajar, mata, aj..."


## **4. Topic Modelling**

In [ ]:
import gensim
import gensim.corpora as corpora
from gensim.models import LdaModel

# Membuat Dictionary (Kamus kata)
id2word = corpora.Dictionary(df['teks_bersih'])

# Filter buang kata yang muncul di kurang dari 2 dokumen atau lebih dari 90% dokumen
id2word.filter_extremes(no_below=2, no_above=0.9)

# Membuat Corpus
texts = df['teks_bersih'].tolist()
corpus = [id2word.doc2bow(text) for text in texts]

# Menentukan jumlah topik
num_topics = 10

print(f"Persiapan data selesai! Corpus berisi {len(corpus)} dokumen.")

Persiapan data selesai! Corpus berisi 235 dokumen.


## **5. Looping & Evaluasi**

In [ ]:
import pandas as pd
import numpy as np
import random
import os
import time
from gensim.models import LdaModel
from gensim.models import CoherenceModel

# Seeds
def set_global_seed(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)

# Menghitung Topic Diversity
def calculate_topic_diversity(model, topk=10):
    topics = model.show_topics(num_topics=-1, num_words=topk, formatted=False)
    unique_words = set()
    total_words = 0

    for topic_id, topic_words in topics:
        for word, weight in topic_words:
            unique_words.add(word)
            total_words += 1

    if total_words == 0:
        return 0
    return len(unique_words) / total_words

# --- PROSES LOOPING 10 RUN ---

seeds = [10, 20, 30, 40, 50, 60, 70, 80, 90, 100]
results = []

print("Memulai pengujian 10 run untuk Model LDA...")

for run_id, seed in enumerate(seeds, start=1):
    print(f"Menjalankan Run {run_id}/10 (Seed: {seed})...", end=" ")

    # 1. Catat waktu mulai per iterasi
    waktu_mulai_run = time.time()

    # 2. Kunci Seed
    set_global_seed(seed)

    # 3. Bangun Model LDA
    lda_model = LdaModel(
        corpus=corpus,
        id2word=id2word,
        num_topics=num_topics,
        random_state=seed,
        update_every=1,
        chunksize=100,
        passes=10,
        alpha='auto',
        per_word_topics=True
    )

    # 4. Hitung Metrik Evaluasi
    cm_cv = CoherenceModel(model=lda_model, texts=texts, dictionary=id2word, coherence='c_v')
    score_coherence = cm_cv.get_coherence()

    cm_npmi = CoherenceModel(model=lda_model, texts=texts, dictionary=id2word, coherence='c_npmi')
    score_npmi = cm_npmi.get_coherence()

    score_diversity = calculate_topic_diversity(lda_model, topk=10)
    score_quality = score_diversity * score_npmi

    # 5. Hitung Waktu Komputasi per Iterasi
    waktu_selesai_run = time.time()
    waktu_per_iterasi = waktu_selesai_run - waktu_mulai_run

    # Cetak hasil waktu per iterasi secara real-time
    print(f"-> Selesai dalam {waktu_per_iterasi:.4f} detik")

    # 6. Simpan hasil ke dalam daftar
    results.append({
        'Run': run_id,
        'Seed': seed,
        'Coherence (C_v)': score_coherence,
        'NPMI': score_npmi,
        'Diversity': score_diversity,
        'Topic Quality': score_quality,
        'Waktu per Iterasi (detik)': round(waktu_per_iterasi, 4)
    })

# --- OUTPUT HASIL ---
df_results = pd.DataFrame(results)

print("\n=== HASIL EVALUASI LDA (10 RUN) ===")
print(df_results.to_string(index=False))

# Simpan ke CSV
file_name = 'evaluasi_lda_10run.csv'
df_results.to_csv(file_name, index=False)
print(f"\nProses selesai! File tersimpan sebagai: {file_name}")

Memulai pengujian 10 run untuk Model LDA...
Menjalankan Run 1/10 (Seed: 10)... -> Selesai dalam 3.2874 detik
Menjalankan Run 2/10 (Seed: 20)... -> Selesai dalam 2.2531 detik
Menjalankan Run 3/10 (Seed: 30)... -> Selesai dalam 2.1747 detik
Menjalankan Run 4/10 (Seed: 40)... -> Selesai dalam 2.2286 detik
Menjalankan Run 5/10 (Seed: 50)... -> Selesai dalam 2.2541 detik
Menjalankan Run 6/10 (Seed: 60)... -> Selesai dalam 3.4451 detik
Menjalankan Run 7/10 (Seed: 70)... -> Selesai dalam 2.2496 detik
Menjalankan Run 8/10 (Seed: 80)... -> Selesai dalam 2.2503 detik
Menjalankan Run 9/10 (Seed: 90)... -> Selesai dalam 2.2324 detik
Menjalankan Run 10/10 (Seed: 100)... -> Selesai dalam 2.1747 detik

=== HASIL EVALUASI LDA (10 RUN) ===
 Run  Seed  Coherence (C_v)      NPMI  Diversity  Topic Quality  Waktu per Iterasi (detik)
   1    10         0.368134 -0.102327       0.71      -0.072652                     3.2874
   2    20         0.368310 -0.152549       0.79      -0.120514                     2

In [ ]:
import pandas as pd

kolom_metrik = ['Coherence (C_v)', 'NPMI', 'Diversity', 'Topic Quality', 'Waktu per Iterasi (detik)']

ringkasan = []

for metrik in kolom_metrik:
    # 1. Menghitung Titik Tengah (Rata-rata / Mean)
    titik_tengah = df_results[metrik].mean()

    # 2. Menghitung Berdasarkan Nilai Mutlak (Min/Max)
    batas_bawah_mutlak = df_results[metrik].min()
    batas_atas_mutlak = df_results[metrik].max()

    # 3. Menghitung Berdasarkan Standar Deviasi
    std_dev = df_results[metrik].std()
    batas_bawah_std = titik_tengah - std_dev
    batas_atas_std = titik_tengah + std_dev

    # Memasukkan hasil ke dalam daftar
    ringkasan.append({
        'Metrik': metrik,
        'Titik Tengah (Mean)': titik_tengah,
        'Bawah (Min)': batas_bawah_mutlak,
        'Atas (Max)': batas_atas_mutlak,
        'Bawah (Mean - SD)': batas_bawah_std,
        'Atas (Mean + SD)': batas_atas_std,
        'Standar Deviasi': std_dev
    })

# Membuat DataFrame untuk tabel ringkasan
df_ringkasan = pd.DataFrame(ringkasan)

# Membulatkan angka
df_ringkasan = df_ringkasan.round(4)

print("=== DATA UNTUK GRAFIK (PLOT ERROR BARS) ===")
print(df_ringkasan.to_string(index=False))

# Ekspor hasil ringkasan ke CSV
file_ringkasan = 'data_grafik_lda.csv'
df_ringkasan.to_csv(file_ringkasan, index=False)
print(f"\nData ringkasan berhasil disimpan sebagai: {file_ringkasan}")

=== DATA UNTUK GRAFIK (PLOT ERROR BARS) ===
                   Metrik  Titik Tengah (Mean)  Bawah (Min)  Atas (Max)  Bawah (Mean - SD)  Atas (Mean + SD)  Standar Deviasi
          Coherence (C_v)               0.3536       0.3370      0.3683             0.3431            0.3641           0.0105
                     NPMI              -0.1087      -0.1525     -0.0838            -0.1289           -0.0885           0.0202
                Diversity               0.7270       0.6400      0.7900             0.6884            0.7656           0.0386
            Topic Quality              -0.0795      -0.1205     -0.0537            -0.0978           -0.0612           0.0183
Waktu per Iterasi (detik)               2.4550       2.1747      3.4451             1.9724            2.9376           0.4826

Data ringkasan berhasil disimpan sebagai: data_grafik_lda.csv


## **6. Visualisasi**

In [ ]:
import pyLDAvis
import pyLDAvis.gensim_models as gensimvis

pyLDAvis.enable_notebook()

# Menyiapkan data untuk visualisasi
vis_data = gensimvis.prepare(lda_model, corpus, id2word)

# Menampilkan visualisasi
vis_data

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


PreparedData(topic_coordinates=              x         y  topics  cluster       Freq
topic                                                
0     -0.085186  0.010752       1        1  24.582408
1     -0.087580 -0.019073       2        1  21.178636
6     -0.122385  0.020668       3        1  20.693296
3     -0.020253 -0.055473       4        1   8.369006
9     -0.128803  0.081262       5        1   7.751083
4      0.008733 -0.212135       6        1   6.418738
5     -0.080385  0.174981       7        1   4.059168
8      0.062957 -0.168951       8        1   2.819293
2      0.160764  0.086747       9        1   2.108966
7      0.292137  0.081220      10        1   2.019406, topic_info=         Term        Freq       Total Category  logprob  loglift
2        ajar  593.000000  593.000000  Default  30.0000  30.0000
42        uji  343.000000  343.000000  Default  29.0000  29.0000
169   kembang  372.000000  372.000000  Default  28.0000  28.0000
69   learning  221.000000  221.000000  Default  27.0000  27.0000
75      pikir  113.000000  113.000000  Default  26.0000  26.0000
..        ...         ...         ...      ...      ...      ...
338   artikel    4.503810    8.899240  Topic10  -4.7808   3.2213
164      bagi    6.350369   30.765544  Topic10  -4.4372   2.3245
271    maupun    5.044621   14.937847  Topic10  -4.6674   2.8168
63   kategori    7.704177  114.108610  Topic10  -4.2440   1.2070
758      baca    5.094005   31.469816  Topic10  -4.6577   2.0814

[589 rows x 6 columns], token_table=      Topic      Freq        Term
term                             
660       2  0.260026   abstraksi
660      10  0.693403   abstraksi
778       4  0.903067  acceptance
698       2  0.932872         acu
43        1  0.075110         ada
...     ...       ...         ...
90        2  0.265091   wawancara
90        3  0.020392   wawancara
90        4  0.030587   wawancara
90        7  0.081567   wawancara
1054      4  0.943521       wawas

[1053 rows x 3 columns], R=30, lambda_step=0.01, plot_opts={'xlab': 'PC1', 'ylab': 'PC2'}, topic_order=[1, 2, 7, 4, 10, 5, 6, 9, 3, 8])

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
